## Match to OpenAlex

In [1]:
import pandas as pd
import requests
import time
import re
from urllib.parse import quote

df = pd.read_csv('B:\\Semester 4 UU\\thesis-best-paper-trajectories\\data\\cleaned\\huang_awards_pilot.csv')
MAILTO = 'shaheryar.4822@student.uu.se'  # Replace

# ── Helpers ────────────────────────────────────────────────────────────────────

def extract_ss_hash(url):
    """Extract 40-char hex hash from any SS URL format."""
    if not isinstance(url, str) or 'semanticscholar' not in url:
        return None
    match = re.search(r'[a-f0-9]{40}', url)
    return match.group(0) if match else None

def ss_to_doi(ss_hash, retries=2):
    """Call SS API to get DOI from SS paper hash."""
    url = f"https://api.semanticscholar.org/graph/v1/paper/{ss_hash}?fields=externalIds,title,year"
    for _ in range(retries):
        try:
            r = requests.get(url, timeout=10)
            if r.status_code == 200:
                data = r.json()
                doi = data.get('externalIds', {}).get('DOI')
                return doi, data.get('title'), data.get('year')
            elif r.status_code == 429:
                time.sleep(5)
        except:
            time.sleep(2)
    return None, None, None

def openalex_by_doi(doi):
    """Direct DOI lookup in OpenAlex - near 100% when DOI exists."""
    url = f"https://api.openalex.org/works/https://doi.org/{doi}"
    try:
        r = requests.get(url, params={'mailto': MAILTO}, timeout=10)
        if r.status_code == 200:
            return r.json()
    except:
        pass
    return None

def openalex_by_title(title, year):
    """Title+year search in OpenAlex (fallback for Google Scholar rows)."""
    # Clean title: remove special chars that break search
    clean = re.sub(r"['\"\-\/\\]", ' ', title).strip()
    clean = re.sub(r'\s+', ' ', clean)
    params = {
        'search': clean,
        'filter': f'publication_year:{year}',
        'per-page': 3,
        'mailto': MAILTO
    }
    try:
        r = requests.get('https://api.openalex.org/works', params=params, timeout=10)
        if r.status_code == 200:
            results = r.json().get('results', [])
            if results:
                return results[0]  # Top result
    except:
        pass
    return None

# ── Main matching loop ──────────────────────────────────────────────────────────

records = []

for i, row in df.iterrows():
    url = str(row.get('paper_url', ''))
    title = str(row.get('paper_title', ''))
    year = int(row.get('year', 0))
    result = {
        'year': year, 'conference': row['conference'],
        'paper_title': title, 'paper_url': url,
        'authors': row.get('authors', ''),
        'openalex_id': None, 'doi': None,
        'match_route': None, 'oa_title': None
    }

    oa_work = None

    # ── Route A: Semantic Scholar hash ──
    ss_hash = extract_ss_hash(url)
    if ss_hash:
        doi, ss_title, ss_year = ss_to_doi(ss_hash)
        time.sleep(0.15)  # SS rate limit: ~100 req/s with API key, ~10 without

        if doi:
            oa_work = openalex_by_doi(doi)
            result['doi'] = doi
            result['match_route'] = 'SS→DOI→OA'

        if not oa_work:
            # Route C: SS found paper but no DOI → title fallback
            search_title = ss_title or title
            oa_work = openalex_by_title(search_title, year)
            result['match_route'] = 'SS→title→OA'

    # ── Route B: Google Scholar → title search ──
    else:
        oa_work = openalex_by_title(title, year)
        result['match_route'] = 'GS→title→OA'

    # ── Store result ──
    if oa_work:
        result['openalex_id'] = oa_work.get('id')
        result['oa_title'] = oa_work.get('title')
        result['authorships'] = str(oa_work.get('authorships', []))
    
    records.append(result)

    # Progress
    if i % 50 == 0:
        matched = sum(1 for r in records if r['openalex_id'])
        print(f"[{i}/{len(df)}] Matched: {matched} | Route A: {sum(1 for r in records if r.get('match_route','').startswith('SS→DOI'))}")
    
    time.sleep(0.1)  # OpenAlex polite rate limit

# ── Save ────────────────────────────────────────────────────────────────────────
out = pd.DataFrame(records)
out.to_csv('B:\\Semester 4 UU\\thesis-best-paper-trajectories\\data\\matched\\huang_matched_openalex.csv', index=False)

matched = out['openalex_id'].notna().sum()
print(f"\n{'='*50}")
print(f"Total: {len(out)} | Matched: {matched} ({matched/len(out)*100:.1f}%)")
print(out.groupby('match_route')['openalex_id'].apply(lambda x: x.notna().sum()))


[0/912] Matched: 1 | Route A: 1
[50/912] Matched: 51 | Route A: 39
[100/912] Matched: 101 | Route A: 71
[150/912] Matched: 151 | Route A: 107
[200/912] Matched: 200 | Route A: 147
[250/912] Matched: 248 | Route A: 182
[300/912] Matched: 297 | Route A: 216
[350/912] Matched: 347 | Route A: 254
[400/912] Matched: 396 | Route A: 293
[450/912] Matched: 444 | Route A: 321
[500/912] Matched: 490 | Route A: 352
[550/912] Matched: 536 | Route A: 383
[600/912] Matched: 585 | Route A: 418
[650/912] Matched: 632 | Route A: 442
[700/912] Matched: 681 | Route A: 471
[750/912] Matched: 731 | Route A: 500
[800/912] Matched: 780 | Route A: 530
[850/912] Matched: 829 | Route A: 557
[900/912] Matched: 879 | Route A: 584

Total: 912 | Matched: 890 (97.6%)
match_route
GS→title→OA     65
SS→DOI→OA      592
SS→title→OA    233
Name: openalex_id, dtype: int64


#### Check

In [2]:
from rapidfuzz import fuzz
# Flag low-confidence matches
out['title_similarity'] = out.apply(
    lambda r: fuzz.ratio(str(r['paper_title']).lower(), str(r['oa_title']).lower())
    if r['match_route'] != 'SS→DOI→OA' else 100, axis=1
)
# Anything below 85 = suspicious
out['low_confidence'] = out['title_similarity'] < 85
print(out['low_confidence'].sum(), "flagged for manual review")


91 flagged for manual review


#### Mannual Checkups

In [3]:
unmatched = out[out['openalex_id'].isna()]
unmatched[['year','conference','paper_title']].to_csv('B:\\Semester 4 UU\\thesis-best-paper-trajectories\\data\\matched\\unmatched_manual.csv', index=False)
